# 03 — Wikidata SPARQL Entity Grounding

Resolve FB15k-237 MIDs to human-readable labels and descriptions via Wikidata SPARQL, with local disk caching.

**No API key required.**

In [ ]:
import sys; sys.path.insert(0, '..')
import time, random
from src.wikidata.sparql import WikidataGrounder
from src.wikidata.cache import SPARQLCache
from src.data.fb15k237 import FB15k237Dataset
from src.config import Settings

settings = Settings()
print(f'SPARQL endpoint: {settings.wikidata_sparql_url}')
print(f'Cache dir      : {settings.cache_dir}')

## 1. Initialise Grounder & Cache

In [ ]:
cache    = SPARQLCache(cache_dir=settings.cache_dir)
grounder = WikidataGrounder(
    sparql_url=settings.wikidata_sparql_url,
    user_agent=settings.wikidata_user_agent,
    cache=cache,
)
print('Grounder ready.')

## 2. Resolve a Single MID

In [ ]:
mid = '/m/0d6lp'
t0  = time.time()
res = grounder.ground(mid)
elapsed = time.time()-t0
print(f'MID  : {mid}')
print(f'Label: {res.get("label","N/A")}')
print(f'Desc : {res.get("description","N/A")}')
print(f'Time : {elapsed:.3f}s')

## 3. Cache Hit vs Miss Latency

In [ ]:
t0 = time.time()
grounder.ground(mid)
cache_time = time.time()-t0
print(f'Network: {elapsed*1000:.0f}ms  Cache: {cache_time*1000:.1f}ms  Speedup: {elapsed/max(cache_time,1e-6):.0f}x')

## 4. Batch Resolve Sample Entities

In [ ]:
dataset = FB15k237Dataset(data_dir=settings.data_dir)
dataset.load()
random.seed(settings.random_seed)
all_ents = list({h for h,r,t in dataset.test}|{t for h,r,t in dataset.test})
sample   = random.sample(all_ents, min(10, len(all_ents)))
resolved = grounder.batch_ground(sample)
print(f'{"MID":<20}{"Label":<30}{"Description":<50}')
print('-'*100)
for mid,info in resolved.items():
    print(f'{mid:<20}{info.get("label","N/A")[:28]:<30}{info.get("description","N/A")[:48]:<50}')

## 5. Cache Statistics

In [ ]:
stats = cache.stats()
print(f'Cached entries: {stats.get("total","N/A")}')
print(f'Cache size    : {stats.get("size_mb","N/A")} MB')

## 6. Full Triple Grounding

In [ ]:
h,r,t = dataset.test[0]
h_info = grounder.ground(h)
t_info = grounder.ground(t)
print('=== Grounded Triple ===')
print(f'Head : {h} -> {h_info.get("label","?")}')
print(f'       "{h_info.get("description","")[:80]}"')
print(f'Rel  : {r}')
print(f'Tail : {t} -> {t_info.get("label","?")}')
print(f'       "{t_info.get("description","")[:80]}"')